In [ ]:
# Copyright (c) 2025 ByteDance Ltd. and/or its affiliates

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http:www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Timer-S1 on GIFT-EVAL Benchmark

Evaluate the Timer-S1 model on the GIFT-EVAL benchmark with quantile forecasting.

Before running, make sure the `GIFT_EVAL` environment variable points to your dataset directory.

In [ ]:
! pip install torch accelerate transformers~=4.57.1

## 2. Configuration

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

model_name = "Timer-S1"

# Datasets to evaluate
short_datasets = (
    "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly "
    "electricity/15T electricity/H electricity/D electricity/W "
    "solar/10T solar/H solar/D solar/W hospital covid_deaths "
    "us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W "
    "temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D "
    "car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W "
    "LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H "
    "M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W "
    "jena_weather/10T jena_weather/H jena_weather/D "
    "bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H "
    "bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
)
med_long_datasets = (
    "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H "
    "LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H "
    "ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H "
    "bitbrains_fast_storage/5T bitbrains_rnd/5T "
    "bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
)

all_datasets = sorted(set(short_datasets.split()) | set(med_long_datasets.split()))
dataset_properties_map = json.load(open("dataset_properties.json"))

output_dir = f"../results/{model_name}"
os.makedirs(output_dir, exist_ok=True)

csv_file_path = os.path.join(output_dir, "all_results.csv")
print(f"Results will be saved to: {csv_file_path}")

## 3. Metrics

In [ ]:
from gluonts.ev.metrics import (
    MAE, MAPE, MASE, MSE, MSIS, ND, NRMSE, RMSE, SMAPE,
    MeanWeightedSumQuantileLoss,
)

metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## 4. Timer-S1 Predictor

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, set_seed
from tqdm.auto import tqdm
from gluonts.itertools import batcher
from gluonts.transform import LastValueImputation
from gluonts.model.forecast import QuantileForecast

set_seed(1)

QUANTILE_KEYS = [str(q) for q in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]]
MAX_CONTEXT_LENGTH = 11520   # maximum input sequence length fed to the model


class TimerS1QuantilePredictor:
    """Wrapper around Timer-S1 for GIFT-EVAL quantile evaluation."""

    def __init__(self, prediction_length: int, device_map="auto"):
        self.prediction_length = prediction_length
        self.model = AutoModelForCausalLM.from_pretrained(
            "bytedance-research/Timer-S1",
            trust_remote_code=True,
            device_map=device_map,
        )
        self.model.config.use_cache = False

    # ── helpers ──────────────────────────────────────────────────────────────

    @staticmethod
    def _left_pad_and_stack(tensors):
        max_len = max(len(t) for t in tensors)
        padded = []
        for t in tensors:
            pad = torch.full((max_len - len(t),), fill_value=torch.nan)
            padded.append(torch.cat([pad, t]))
        return torch.stack(padded)

    def _prepare_context(self, context):
        if isinstance(context, list):
            context = self._left_pad_and_stack(context)
        if context.ndim == 1:
            context = context.unsqueeze(0)
        return context

    @staticmethod
    def _impute_nan(batch_x: torch.Tensor) -> torch.Tensor:
        arr = batch_x.numpy()
        rows = []
        for row in arr:
            rows.append(LastValueImputation()(row))
        return torch.tensor(np.vstack(rows))

    # ── main predict ─────────────────────────────────────────────────────────

    def predict(self, test_data_input, batch_size: int = 256):
        try:
            first_device = next(self.model.parameters()).device
        except StopIteration:
            first_device = torch.device("cpu")

        forecasts = []
        for batch in tqdm(batcher(test_data_input, batch_size=batch_size)):
            context = [torch.tensor(entry["target"]) for entry in batch]
            batch_x = self._prepare_context(context)

            if batch_x.shape[-1] > MAX_CONTEXT_LENGTH:
                batch_x = batch_x[..., -MAX_CONTEXT_LENGTH:]

            if torch.isnan(batch_x).any():
                batch_x = self._impute_nan(batch_x)

            batch_x = batch_x.to(first_device)

            with torch.autocast(device_type=first_device.type, dtype=torch.bfloat16):
                outputs = self.model.generate(
                    batch_x,
                    max_new_tokens=self.prediction_length,
                    revin=True,
                )  # shape: [batch, num_quantiles, pred_len]

            outputs = outputs.detach().cpu().float().numpy()

            for item, ts in zip(outputs, batch):
                forecast_start = ts["start"] + len(ts["target"])
                forecasts.append(
                    QuantileForecast(
                        forecast_arrays=item,
                        forecast_keys=QUANTILE_KEYS,
                        start_date=forecast_start,
                    )
                )

        return forecasts

## 5. Evaluation

In [ ]:
import csv
import logging

from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

# Suppress noisy GluonTS warning
_gts_logger = logging.getLogger("gluonts.model.forecast")
_gts_logger.addFilter(type("F", (logging.Filter,), {
    "filter": lambda self, r: "mean prediction is not stored" not in r.getMessage()
})())

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

# Write CSV header
with open(csv_file_path, "w", newline="") as f:
    csv.writer(f).writerow([
        "dataset", "model",
        "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]",
        "eval_metrics/MASE[0.5]", "eval_metrics/MAPE[0.5]", "eval_metrics/sMAPE[0.5]",
        "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]",
        "eval_metrics/ND[0.5]", "eval_metrics/mean_weighted_sum_quantile_loss",
        "domain", "num_variates",
    ])

med_long_set = set(med_long_datasets.split())

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")

    for term in ("short", "medium", "long"):
        if term in ("medium", "long") and ds_name not in med_long_set:
            continue

        if "/" in ds_name:
            ds_key = pretty_names.get(ds_name.split("/")[0].lower(), ds_name.split("/")[0].lower())
            ds_freq = ds_name.split("/")[1]
        else:
            ds_key = pretty_names.get(ds_name.lower(), ds_name.lower())
            ds_freq = dataset_properties_map[ds_key]["frequency"]

        ds_config = f"{ds_key}/{ds_freq}/{term}"

        to_univariate = (
            Dataset(name=ds_name, term=term, to_univariate=False).target_dim > 1
        )
        dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
        season_length = get_seasonality(dataset.freq)
        print(f"Dataset size: {len(dataset.test_data)}")

        predictor = TimerS1QuantilePredictor(
            prediction_length=dataset.prediction_length,
            device_map="auto",
        )

        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=256,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as f:
            csv.writer(f).writerow([
                ds_config, model_name,
                res["MSE[mean]"][0], res["MSE[0.5]"][0], res["MAE[0.5]"][0],
                res["MASE[0.5]"][0], res["MAPE[0.5]"][0], res["sMAPE[0.5]"][0],
                res["MSIS"][0], res["RMSE[mean]"][0], res["NRMSE[mean]"][0],
                res["ND[0.5]"][0], res["mean_weighted_sum_quantile_loss"][0],
                dataset_properties_map[ds_key]["domain"],
                dataset_properties_map[ds_key]["num_variates"],
            ])
        print(f"Results for {ds_name} have been written to {csv_file_path}")

## 6. Results

In [ ]:
import pandas as pd

df = pd.read_csv(f"../results/{model_name}/all_results.csv")
df